# **Project 2 - Sales Analysis - ETL**

## Objectives

- Extract data from provided CSV files
- Clean it 
- Apply feature engineering if necessary 
- Remove any unnecessary columns
- Save to a new CSV file


## Inputs

CSV files provided:

stores data-set.csv
sales data-set.csv
features data set.csv

Renamed files to a uniform naming convention:

Sales_Features_DataSet.csv
Sales_DataSet.csv
Sales_Stores_DataSet.csv

Note: original files are stored in Data/OriginalFiles


## Outputs

CSV file created from ETL etc stored in Data:

Sales_Combined_DataSet.csv

## Additional Comments

Developed an experimental ETL library which is in this project (modETL_library.py) 
Has lots of cool features so will be interesting to see how it works "in the field"



Used various AI tools to help with the ETL process:
- ChatGPT
- GitHub Copilot. 

See Documents/What_AI_Used_For.md for more details.

## Initalise Working Environment

In [1]:
#import libraries
import os
import numpy as np
import pandas as pd

#below solution provided by chatGPT 
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))
    
import modGlobal
import modETL_Library as modETL
#end solution provided by chatGPT

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [2]:
#DataFrame vars for ETL
dfSales_Features_DataSet = None
dfSales_Stores_DataSet = None
dfSales_DataSet = None
dfSales_Combined_DataSet = None

#stores current directory
strCurrentDir = ""

#other vars
intMissingPercent = 0
intMissing = 0
dictDataFrames = dict()
lstColumns = list()

## Set Current Directory To Base Project Directory

In [3]:
#get project directory - default is jupyter notebook sub folder as that is where this file is located!
#so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

#get current folder
strCurrentDir = os.getcwd()

#is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
   #get current working directory and move back one to the project root path
   strCurrentDir =  os.path.normpath(os.getcwd() + os.sep + os.pardir)
   os.chdir(os.path.dirname(strCurrentDir))
   #change directory
   os.chdir(strCurrentDir)

#confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/rogerwilliams/Projects/Python/CourseProjects/Project2-SalesAnalysis


# Section 1 - Combination

- Read csv files
- Combine into one DataFrame

## Read csv Files Into Variable For Processing

In [4]:
#read csv files into DataFrames
#ETL library returns a dictionary of all files in the folder with the attribute name
#set to the actual csv filename
dictDataFrames = modETL.funcReadVisualisationFilesReturnDictionary()
dfSales_DataSet = dictDataFrames["Sales_DataSet_Visualisation.csv"]
dfSales_Features_DataSet = dictDataFrames["Features_DataSet_Visualisation.csv"]
dfSales_Stores_DataSet = dictDataFrames["Stores_DataSet_Visualisation.csv"]


4 csv Files Read Into DataFrames

DataFrames Created:
Sales_Combined_DataSet_Visualisation.csv
Features_DataSet_Visualisation.csv
Sales_DataSet_Visualisation.csv
Stores_DataSet_Visualisation.csv




## Combine The Data Frames!

## How The DataFrames Are Related

_"primary key"_  
dfSales_Stores_DataSet Has One Common Column: Store  
dfSales_DataSet Has One Common Column With dfSales_Stores_DataSet: Store  
dfSales_Features_DataSet Has One Common Column With dfSales_Stores_DataSet: Store  

_Columns Shared Between dfSales_DataSet And dfSales_Features_DataSet:_    
Store, Date

Combined Columns To Use For Analysis:  
Store: int64  
Date: datetime64[us]  
Size: int64  
Temperature: float64  
Dept: int64. 
Weekly_Sales: float64  
IsHoliday: bool  
MarkDown1: float64  
MarkDown2: float64  
MarkDown3: float64  
MarkDown4: float64  
MarkDown5: float64  
StoreType: int64 <- Feature Engineering addition to Sales_Stores_DataSet.csv>  


In [5]:
#first convert the Date columns to datetime format
dfSales_DataSet["Date"] = pd.to_datetime(dfSales_DataSet["Date"], format="%d/%m/%Y")
dfSales_Features_DataSet["Date"] = pd.to_datetime(dfSales_Features_DataSet["Date"], format="%d/%m/%Y")

#combine DataFrames
dfSales_Combined_DataSet = pd.merge(dfSales_DataSet, dfSales_Features_DataSet, on=['Store', 'Date'], how='inner')

# Merge Sales and Features
dfSales_Combined_DataSet = pd.merge(
    dfSales_DataSet,
    dfSales_Features_DataSet,
    on=["Store", "Date"],
    how="inner",
    indicator = True
)

# Merge the result with Stores
dfSales_Combined_DataSet = pd.merge(
    dfSales_Combined_DataSet,
    dfSales_Stores_DataSet,
    on="Store",
    how="inner"
)

#show results need huge value so we can check data against the original data sets
dfSales_Combined_DataSet.head(100000)



,Unnamed: 0_x,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Unnamed: 0_y,Temperature,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,IsHoliday_y,Unemployment,_merge,Unnamed: 0,Size,Store_Type
0,0,1,1,2010-02-05,24924.50,False,0,42.31,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1
1,1,1,1,2010-02-12,46039.49,True,1,38.51,0.0,0.0,0.0,0.0,0.0,True,8.106,both,0,151315,1
2,2,1,1,2010-02-19,41595.55,False,2,39.93,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1
3,3,1,1,2010-02-26,19403.54,False,3,46.63,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1
4,4,1,1,2010-03-05,21827.90,False,4,46.50,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99995,11,17,2010-04-23,13984.63,False,1831,68.37,0.0,0.0,0.0,0.0,0.0,False,7.343,both,10,207499,1
99996,99996,11,17,2010-04-30,10486.69,False,1832,71.13,0.0,0.0,0.0,0.0,0.0,False,7.343,both,10,207499,1
99997,99997,11,17,2010-05-07,12840.19,False,1833,75.57,0.0,0.0,0.0,0.0,0.0,False,7.343,both,10,207499,1
99998,99998,11,17,2010-05-14,11494.95,False,1834,77.64,0.0,0.0,0.0,0.0,0.0,False,7.343,both,10,207499,1


## Observations - Sales_Combined_DataSet

Checks:

Checked Column value change for Markdown1 at page: 81 to check if had value: 10382.9
as per the raw csv file.

Checked column value change for Store at page: 2049 to check if StoreType column
was correctly showing the value 2 when the Store column value was 3 as in the 
dfSales_Store_DataSet DataFrame Store value 3 = store type B which in the new column
added during data transformation in Notebook_ETL_Sales_Stores_DataSet.ipynb is 2

All values found as expected.

Note: In Features data set.csv there is 7 months of data NOT in sales data-set.csv so needed 
      to filter that out of the results

## Observations - Sales_Combined_DataSet
Check new DataFrame schema

In [6]:
#check new DataFrame schema
dfSales_Combined_DataSet.dtypes


Unnamed: 0_x             int64
Store                    int64
Dept                     int64
Date            datetime64[us]
Weekly_Sales           float64
IsHoliday_x               bool
Unnamed: 0_y             int64
Temperature            float64
MarkDown1              float64
MarkDown2              float64
MarkDown3              float64
MarkDown4              float64
MarkDown5              float64
IsHoliday_y               bool
Unemployment           float64
_merge                category
Unnamed: 0               int64
Size                     int64
Store_Type               int64
dtype: object

## Observations - Sales_Combined_DataSet

Columns count correct, need to rename some other like Unnamed will simply be excluded from
addition to the cleaned csv

---

# Section 2 - Transformation

- Rename any columns if necessary

Columns To Rename:  
IsHoliday_x -> IsHoliday  
Size -> Store_Size  



## Observations - Rename Columns

In [7]:
#rename columns: IsHoliday_x -> IsHoliday   Size_x -> Store_Size
dfSales_Combined_DataSet.rename(columns={'IsHoliday_x': 'IsHoliday', 'Size': 'Store_Size'}, inplace=True)

#check results
dfSales_Combined_DataSet.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 19 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Unnamed: 0_x  421570 non-null  int64         
 1   Store         421570 non-null  int64         
 2   Dept          421570 non-null  int64         
 3   Date          421570 non-null  datetime64[us]
 4   Weekly_Sales  421570 non-null  float64       
 5   IsHoliday     421570 non-null  bool          
 6   Unnamed: 0_y  421570 non-null  int64         
 7   Temperature   421570 non-null  float64       
 8   MarkDown1     421570 non-null  float64       
 9   MarkDown2     421570 non-null  float64       
 10  MarkDown3     421570 non-null  float64       
 11  MarkDown4     421570 non-null  float64       
 12  MarkDown5     421570 non-null  float64       
 13  IsHoliday_y   421570 non-null  bool          
 14  Unemployment  421570 non-null  float64       
 15  _merge        421570 non-nul

## Observations - Combined_DataSet

Schema as expected

## Check For Missing Values (Paranoia)

In [8]:
#get total missing values
intMissing = dfSales_Combined_DataSet.isnull().sum()

#calculate percent missing
intMissingPercent = ( intMissing / len(dfSales_Combined_DataSet) * 100)
dfCheck = pd.DataFrame( {"Missing Values": intMissing, "%" : intMissingPercent})

#show results
if dfCheck["Missing Values"].sum() !=0:
   print( dfCheck[dfCheck["Missing Values"] > 0])
else:
   print( "No Missing Values")     


No Missing Values


## Observations - Combined_DataSet

No Missing Values - Phew!

---

## Data Transformations/Feature Engineering

Had Mad Scientist idea:

Create a column with just year and month in it, if I get time (and if its actually feasible)
Might be able to knock some visualsations by month...

In [9]:
#create a column with JUST year and month as integer
dfSales_Combined_DataSet["YearMonth"] = dfSales_Combined_DataSet["Date"].dt.strftime('%Y%m').astype(int)

#check results
dfSales_Combined_DataSet.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 20 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Unnamed: 0_x  421570 non-null  int64         
 1   Store         421570 non-null  int64         
 2   Dept          421570 non-null  int64         
 3   Date          421570 non-null  datetime64[us]
 4   Weekly_Sales  421570 non-null  float64       
 5   IsHoliday     421570 non-null  bool          
 6   Unnamed: 0_y  421570 non-null  int64         
 7   Temperature   421570 non-null  float64       
 8   MarkDown1     421570 non-null  float64       
 9   MarkDown2     421570 non-null  float64       
 10  MarkDown3     421570 non-null  float64       
 11  MarkDown4     421570 non-null  float64       
 12  MarkDown5     421570 non-null  float64       
 13  IsHoliday_y   421570 non-null  bool          
 14  Unemployment  421570 non-null  float64       
 15  _merge        421570 non-nul

## Observations - Combined_DataSet

Schema as expected

## Check Data In New Column

In [10]:
#check data
dfSales_Combined_DataSet.head(50)

,Unnamed: 0_x,Store,Dept,Date,Weekly_Sales,IsHoliday,Unnamed: 0_y,Temperature,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,IsHoliday_y,Unemployment,_merge,Unnamed: 0,Store_Size,Store_Type,YearMonth
0,0,1,1,2010-02-05,24924.50,False,0,42.31,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1,201002
1,1,1,1,2010-02-12,46039.49,True,1,38.51,0.0,0.0,0.0,0.0,0.0,True,8.106,both,0,151315,1,201002
2,2,1,1,2010-02-19,41595.55,False,2,39.93,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1,201002
3,3,1,1,2010-02-26,19403.54,False,3,46.63,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1,201002
4,4,1,1,2010-03-05,21827.90,False,4,46.50,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1,201003
5,5,1,1,2010-03-12,21043.39,False,5,57.79,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1,201003
6,6,1,1,2010-03-19,22136.64,False,6,54.58,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1,201003
7,7,1,1,2010-03-26,26229.21,False,7,51.45,0.0,0.0,0.0,0.0,0.0,False,8.106,both,0,151315,1,201003
8,8,1,1,2010-04-02,57258.43,False,8,62.27,0.0,0.0,0.0,0.0,0.0,False,7.808,both,0,151315,1,201004
9,9,1,1,2010-04-09,42960.91,False,9,65.86,0.0,0.0,0.0,0.0,0.0,False,7.808,both,0,151315,1,201004


## Observations - Combined_DataSet

Data as expected

## Save Combined DataFrame To CSV File

In [11]:
print(dfSales_Combined_DataSet["Date"])

0        2010-02-05
1        2010-02-12
2        2010-02-19
3        2010-02-26
4        2010-03-05
            ...    
421565   2012-09-28
421566   2012-10-05
421567   2012-10-12
421568   2012-10-19
421569   2012-10-26
Name: Date, Length: 421570, dtype: datetime64[us]


In [12]:
#create list of columns new and old JUST what is actually needed
lstColumns = [
                #columns to use in the cleaned file (excludes any index)
                "Store","Dept","Date","Weekly_Sales","IsHoliday",
                "MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5",
                "Store_Type","Store_Size","Temperature","Unemployment",
                "YearMonth"
               ]

#create new DataFrame for cleaned data
dfCleaned = dfSales_Combined_DataSet[lstColumns].copy()
dfCleaned.attrs["name"] = "Sales_Combined_DataSet_Working"

modETL.funcSaveDataFrameToCleanedFile(dfCleaned)

#Note: excluded column: Dept as not in the hypothesis analysis which is a nice way
#      avoiding the fact I don't know just how to plot a chart with 98 departments
#      and have it readable!

Saved: Sales_Combined_DataSet To Visualisation Folder


# Next Steps

End of ETL process. Now visualising the hypothesis can begin!

See notebook:
Notebook_Visualisations.jpynb
